# WakeRef — Convertir le modèle en ONNX + pousser sur Hugging Face

À faire APRÈS l'entraînement. Tu auras besoin :
- du fichier **`whisper-wakeref.zip`** (le modèle, ~290 Mo),
- d'un compte **huggingface.co** + un **token "Write"** (Settings → Access Tokens).

Lance les cellules dans l'ordre (▶). Le GPU n'est pas nécessaire ici.


## 1. Installer Optimum (l'outil de conversion ONNX) — ~2 min

In [ ]:
!pip -q install "optimum[onnxruntime]"
print('OK')


## 2. Envoyer le modèle (`whisper-wakeref.zip`) et le décompresser
Quand le bouton apparaît, choisis ton zip du modèle.

In [ ]:
from google.colab import files
files.upload()
!unzip -q -o whisper-wakeref*.zip -d model_pt
import os
PT = next(r for r,_,fs in os.walk('model_pt') if 'config.json' in fs)
print('modele PyTorch :', PT)


## 3. Convertir en ONNX (~1-2 min)

In [ ]:
!optimum-cli export onnx -m "$PT" --task automatic-speech-recognition-with-past onnx_out
import os; print('produit :', os.listdir('onnx_out'))


## 4. Réorganiser au format de l'app (+ copier le tokenizer)
Configs + tokenizer à la racine, `.onnx` dans `onnx/`. Optimum n'exporte PAS le
tokenizer → on le reprend du modèle PyTorch (sinon l'app plante : `tokenizer is null`).

In [ ]:
import os, glob, shutil
os.makedirs('hf/onnx', exist_ok=True)
# .onnx -> hf/onnx/
for f in glob.glob('onnx_out/*.onnx') + glob.glob('onnx_out/*.onnx_data'):
    shutil.move(f, 'hf/onnx/' + os.path.basename(f))
# configs produits par Optimum -> hf/
for f in glob.glob('onnx_out/*'):
    if os.path.isfile(f): shutil.copy(f, 'hf/' + os.path.basename(f))
# tokenizer depuis le modele PyTorch -> hf/
for name in ['tokenizer.json','tokenizer_config.json','vocab.json','merges.txt',
             'normalizer.json','special_tokens_map.json','added_tokens.json']:
    src = os.path.join(PT, name)
    if os.path.exists(src): shutil.copy(src, 'hf/' + name)
OUT = 'hf'
print('racine :', sorted(os.listdir('hf')))
print('onnx/  :', sorted(os.listdir('hf/onnx')))
assert 'tokenizer.json' in os.listdir('hf'), 'tokenizer manquant !'


## 5. Pousser sur Hugging Face
Remplis **TON_USER** et **ton token Write** (entre les guillemets), puis lance.
Le token en clair : ne partage pas le notebook, ou efface-le après.

In [ ]:
USER  = 'TON_USER'                       # ← ton identifiant huggingface.co
TOKEN = 'hf_colle_ton_token_Write'       # ← huggingface.co/settings/tokens (Write)
REPO  = f'{USER}/whisper-wakeref-onnx'
from huggingface_hub import login, HfApi
login(token=TOKEN)                        # authentifie AVANT le push (bloquant)
api = HfApi()
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO)
print('OK pousse :', f'https://huggingface.co/{REPO}')


## C'est fait
Modèle complet (onnx + tokenizer) sur `https://huggingface.co/TON_USER/whisper-wakeref-onnx`.
Recharge `/judge/voix`, choisis le modèle « maison ✨ », précharge, teste.